## BASE

In [ ]:
import torch
import torch.nn as nn
from torchvision import transforms, datasets
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import f1_score
import time
from tqdm.notebook import tqdm
from torch.utils.data import Subset
from collections import defaultdict
import random

In [ ]:
DATA_ROOT = "/home/alex/internship/datasets/aqua20/data/aqua20"
NUM_CLASSES = 20
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RESOLUTION = 252

In [ ]:
transform = transforms.Compose([
    transforms.Resize(RESOLUTION),
    transforms.CenterCrop(RESOLUTION),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

In [ ]:
backbone = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitb14')
backbone.eval()
for p in backbone.parameters():
    p.requires_grad = False
backbone = backbone.to(DEVICE)

In [ ]:
def extract_features(loader, desc="Extracting features"):
    all_feats, all_labels = [], []
    with torch.no_grad():
        for x, y in tqdm(loader, desc=desc, leave=False):
            all_feats.append(backbone(x.to(DEVICE)).cpu())
            all_labels.append(y)
    return torch.cat(all_feats), torch.cat(all_labels)


def train_linear_probe(train_feats, train_labels, test_feats, test_labels,
                       epochs=50, lr=1e-3, eval_every=10):
    head = nn.Linear(768, NUM_CLASSES).to(DEVICE)
    optimizer = torch.optim.Adam(head.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    # Loaders sur features précalculées — tout en RAM, très rapide
    train_feat_loader = DataLoader(
        TensorDataset(train_feats, train_labels), batch_size=256, shuffle=True
    )
    test_feat_loader = DataLoader(
        TensorDataset(test_feats, test_labels), batch_size=256, shuffle=False
    )

    for epoch in tqdm(range(epochs), desc="Training"):
        head.train()
        total_loss, correct, total = 0.0, 0, 0

        for feats, y in train_feat_loader:
            feats, y = feats.to(DEVICE), y.to(DEVICE)
            logits = head(feats)
            loss = criterion(logits, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * len(y)
            correct += (logits.argmax(dim=1) == y).sum().item()
            total += len(y)

        if (epoch + 1) % eval_every == 0 or epoch == epochs - 1:
            head.eval()
            all_preds, all_labels_val = [], []
            with torch.no_grad():
                for feats, y in test_feat_loader:
                    feats = feats.to(DEVICE)
                    preds = head(feats).argmax(dim=1).cpu()
                    all_preds.append(preds)
                    all_labels_val.append(y)

            all_preds      = torch.cat(all_preds).numpy()
            all_labels_val = torch.cat(all_labels_val).numpy()

            f1_macro = f1_score(all_labels_val, all_preds, average="macro")
            f1_weighted = f1_score(all_labels_val, all_preds, average="weighted")

            tqdm.write(
                f"Epoch {epoch+1:>3}/{epochs} | "
                f"Loss: {total_loss/total:.4f} | "
                f"Train Acc: {correct/total*100:.1f}% | "
                f"F1 Macro: {f1_macro*100:.1f}% | "
                f"F1 Weighted: {f1_weighted*100:.1f}%"
            )

    return head

In [ ]:
DISTILLED_PTH = "../logged_files/distillation/aqua20/dinov2_vitb/dinov2_vitb_distill_252_ipc1_augs10_physics/data.pth"
distilled = torch.load(DISTILLED_PTH, map_location=DEVICE)
# distilled est un tensor (N, C, H, W) ou un dict selon ton format
# adapte selon ce que run.sh sauvegarde
print(f"Distilled data keys: {distilled.keys()}")
images_d = distilled["images"].to(DEVICE)   # shape: (20, 3, 196, 196)
labels_d = distilled["labels"].to(DEVICE)
print(labels_d)
from torch.utils.data import TensorDataset
distill_loader = DataLoader(
    TensorDataset(images_d.cpu(), labels_d.cpu()),
    batch_size=20, shuffle=True
)

In [ ]:
test_ds    = datasets.ImageFolder(f"{DATA_ROOT}/test",  transform=transform)
test_loader = DataLoader(test_ds,   batch_size=64, shuffle=False, num_workers=4)

## Baselines



### Full data


In [ ]:
full_train = datasets.ImageFolder(f"{DATA_ROOT}/train", transform=transform)
full_loader = DataLoader(full_train, batch_size=64, shuffle=True, num_workers=4)

In [ ]:
print("Extracting features...")
test_feats,    test_labels    = extract_features(test_loader,    "Test")

In [ ]:
print("\nTraining on full data...")
start_time = time.time()
full_feats,    full_labels    = extract_features(full_loader,    "Full train")
head_full = train_linear_probe(full_feats, full_labels, test_feats, test_labels,
                                epochs=50, eval_every=10)
print(f"Training time: {time.time() - start_time:.6f} seconds")

### Random subset


In [ ]:
ipc = len(images_d) // len(full_train.classes)
rng = random.Random(123)

class_to_indices = defaultdict(list)
for idx, (_, label) in enumerate(full_train.samples):
    class_to_indices[label].append(idx)



stratified_indices = []
for label, indices in class_to_indices.items():
    stratified_indices.extend(rng.sample(indices, ipc))

random_subset = Subset(full_train, stratified_indices)

random_loader = DataLoader(
    random_subset,
    batch_size=len(stratified_indices),
    shuffle=True,
    num_workers=4,
)

In [ ]:
print("\nTraining on random data...")
start_time = time.time()
random_feats, random_labels = extract_features(random_loader, "Distilled")
head_dist = train_linear_probe(random_feats, random_labels, test_feats, test_labels,
                                epochs=50, eval_every=20)
print(f"Training time: {time.time() - start_time:.6f} seconds")

### Random head

In [ ]:
def evaluate_random_head(test_feats, test_labels, feat_dim=768):
    head = nn.Linear(feat_dim, NUM_CLASSES).to(DEVICE)
    head.eval()

    test_feat_loader = DataLoader(
        TensorDataset(test_feats, test_labels), batch_size=256, shuffle=False
    )

    all_preds, all_labels_val = [], []
    with torch.no_grad():
        for feats, y in test_feat_loader:
            feats = feats.to(DEVICE)
            preds = head(feats).argmax(dim=1).cpu()
            all_preds.append(preds)
            all_labels_val.append(y)

    all_preds      = torch.cat(all_preds).numpy()
    all_labels_val = torch.cat(all_labels_val).numpy()

    acc         = (all_preds == all_labels_val).mean()
    f1_macro    = f1_score(all_labels_val, all_preds, average="macro")
    f1_weighted = f1_score(all_labels_val, all_preds, average="weighted")

    print(
        f"Random head baseline | "
        f"Test Acc: {acc*100:.1f}% | "
        f"F1 Macro: {f1_macro*100:.1f}% | "
        f"F1 Weighted: {f1_weighted*100:.1f}%"
    )
    return head

print("\nBaseline: random (untrained) head...")
evaluate_random_head(test_feats, test_labels)

## Distilled data


### No Physics


In [ ]:
NO_PHYSICS_DATAPATH = "../logged_files/distillation/aqua20/dinov2_vitb/dinov2_vitb_distill_252_ipc1_augs10/data.pth"
no_physics_data = torch.load(NO_PHYSICS_DATAPATH, map_location=DEVICE)
# no_physics_data est un tensor (N, C, H, W) ou un dict selon ton format
# adapte selon ce que run.sh sauvegarde
print(f"No physics data keys: {no_physics_data.keys()}")
images_no_physics = no_physics_data["images"].to(DEVICE)   # shape: (20, 3, 196, 196)
labels_no_physics = no_physics_data["labels"].to(DEVICE)
print(labels_no_physics)
from torch.utils.data import TensorDataset
no_physics_loader = DataLoader(
    TensorDataset(images_no_physics.cpu(), labels_no_physics.cpu()),
    batch_size=20, shuffle=True
)

In [ ]:
print("\nTraining on no physics data...")
start_time = time.time()
no_physics_feats, no_physics_labels = extract_features(no_physics_loader, "No Physics")
head_no_physics = train_linear_probe(no_physics_feats, no_physics_labels, test_feats, test_labels,
                                      epochs=200, eval_every=20)
print(f"Training time: {time.time() - start_time:.6f} seconds")

### Physics

In [ ]:
PHYSICS_DATAPATH = "../logged_files/distillation/aqua20/dinov2_vitb/dinov2_vitb_distill_252_ipc1_augs10_physics/data.pth"
physics_data = torch.load(PHYSICS_DATAPATH, map_location=DEVICE)
# physics_data est un tensor (N, C, H, W) ou un dict selon ton format
# adapte selon ce que run.sh sauvegarde
print(f"Physics data keys: {physics_data.keys()}")
images_physics = physics_data["images"].to(DEVICE)   # shape: (20, 3, 196, 196)
labels_physics = physics_data["labels"].to(DEVICE)
print(labels_physics)
from torch.utils.data import TensorDataset
physics_loader = DataLoader(
    TensorDataset(images_physics.cpu(), labels_physics.cpu()),
    batch_size=20, shuffle=True
)

In [ ]:
print("\nTraining on physics data...")
start_time = time.time()
physics_feats, physics_labels = extract_features(physics_loader, "Physics")
head_physics = train_linear_probe(physics_feats, physics_labels, test_feats, test_labels,
                                   epochs=200, eval_every=20)
print(f"Training time: {time.time() - start_time:.6f} seconds")

### SeaThru

In [ ]:
SEATHRU_DATAPATH = "../logged_files/distillation/aqua20/dinov2_vitb/dinov2_vitb_distill_252_ipc1_augs10_seathru_v100_32g/data.pth"
seathru_data = torch.load(SEATHRU_DATAPATH, map_location=DEVICE)
# seathru_data est un tensor (N, C, H, W) ou un dict selon ton format
# adapte selon ce que run.sh sauvegarde
print(f"Seathru data keys: {seathru_data.keys()}")
images_seathru = seathru_data["images"].to(DEVICE)   # shape: (20, 3, 196, 196)
labels_seathru = seathru_data["labels"].to(DEVICE)
print(labels_seathru)
from torch.utils.data import TensorDataset
seathru_loader = DataLoader(
    TensorDataset(images_seathru.cpu(), labels_seathru.cpu()),
    batch_size=20, shuffle=True
)

In [ ]:
print("\nTraining on seathru data...")
start_time = time.time()
seathru_feats, seathru_labels = extract_features(seathru_loader, "Seathru")
head_seathru = train_linear_probe(seathru_feats, seathru_labels, test_feats, test_labels,
                                   epochs=200, eval_every=20)
print(f"Training time: {time.time() - start_time:.6f} seconds")

## Seathru 10ipc

In [ ]:
SEATHRU_10IPC_DATAPATH = "../logged_files/distillation/aqua20/dinov2_vitb/distill_aqua20_h100_seathru_10ipc/data.pth"
seathru_data_10ipc = torch.load(SEATHRU_10IPC_DATAPATH, map_location=DEVICE)
# seathru_data est un tensor (N, C, H, W) ou un dict selon ton format
# adapte selon ce que run.sh sauvegarde
print(f"Seathru data keys: {seathru_data_10ipc.keys()}")
images_seathru_10ipc = seathru_data_10ipc["images"].to(DEVICE)   # shape: (20, 3, 196, 196)
labels_seathru_10ipc = seathru_data_10ipc["labels"].to(DEVICE)
print(labels_seathru_10ipc)
from torch.utils.data import TensorDataset
seathru_loader_10ipc = DataLoader(
    TensorDataset(images_seathru_10ipc.cpu(), labels_seathru_10ipc.cpu()),
    batch_size=20, shuffle=True
)

In [ ]:
print("\nTraining on seathru data...")
start_time = time.time()
seathru_feats_10ipc, seathru_labels_10ipc = extract_features(seathru_loader_10ipc, "Seathru")
head_seathru = train_linear_probe(seathru_feats_10ipc, seathru_labels_10ipc, test_feats, test_labels,
                                   epochs=200, eval_every=20)
print(f"Training time: {time.time() - start_time:.6f} seconds")

### Confusion matrix and classification details

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report


@torch.no_grad()
def get_predictions(head, feats, labels):
    head.eval()
    preds = head(feats.to(DEVICE)).argmax(dim=1).cpu().numpy()
    return preds, labels.numpy()


def plot_confusion_matrix(head, test_feats, test_labels, class_names, normalize="true"):
    preds, true = get_predictions(head, test_feats, test_labels)

    cm = confusion_matrix(
        true, preds, labels=range(len(class_names)), normalize=normalize
    )

    fig, ax = plt.subplots(figsize=(13, 11))
    im = ax.imshow(cm, cmap="viridis", vmin=0, vmax=1 if normalize else None)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    ax.set_xticks(range(len(class_names)))
    ax.set_yticks(range(len(class_names)))
    ax.set_xticklabels(class_names, rotation=90, fontsize=8)
    ax.set_yticklabels(class_names, fontsize=8)
    ax.set_xlabel("Prédit")
    ax.set_ylabel("Vrai")
    ax.set_title(f"Matrice de confusion (normalize={normalize})")

    # annotations (utile car 20 classes restent lisibles)
    thresh = cm.max() / 2.0
    for i in range(len(class_names)):
        for j in range(len(class_names)):
            val = cm[i, j]
            if val > 0:
                ax.text(j, i, f"{val:.2f}" if normalize else f"{int(val)}",
                        ha="center", va="center", fontsize=6,
                        color="white" if val < thresh else "black")

    plt.tight_layout()
    plt.show()

    # rapport precision / recall / f1 par classe
    print(classification_report(true, preds, target_names=class_names, digits=3, zero_division=0))
    return cm

In [ ]:
from PIL import Image


def _load_img(test_ds, idx):
    return Image.open(test_ds.samples[idx][0]).convert("RGB")


def _reference_idx(class_idx, true, preds, exclude=None):
    """Index d'une image de class_idx, de préférence bien classée."""
    pool = np.where(true == class_idx)[0]
    correct = pool[preds[pool] == class_idx]
    pool = correct if len(correct) > 0 else pool      # fallback si aucune correcte
    if exclude is not None:
        pool = pool[pool != exclude]
    return np.random.choice(pool) if len(pool) > 0 else None


def show_misclassified(head, test_feats, test_labels, test_ds, class_names,
                       n=12, target_class=None, cols=4):
    preds, true = get_predictions(head, test_feats, test_labels)
    wrong = np.where(preds != true)[0]

    if target_class is not None:
        wrong = wrong[true[wrong] == target_class]

    if len(wrong) == 0:
        print("Aucune image mal classée pour ce filtre.")
        return

    chosen = np.random.choice(wrong, size=min(n, len(wrong)), replace=False)

    # --- Mode comparaison : triplets (mal classée | vraie classe | classe prédite) ---
    if target_class is not None:
        rows = len(chosen)
        fig, axes = plt.subplots(rows, 3, figsize=(3 * 3, rows * 3))
        axes = np.atleast_2d(axes)

        for r, idx in enumerate(chosen):
            t, p = true[idx], preds[idx]

            # image mal classée
            axes[r, 0].imshow(_load_img(test_ds, idx))
            axes[r, 0].set_title(f"mal classée\nvrai={class_names[t]}\nprédit={class_names[p]}",
                                 fontsize=8, color="darkred")

            # référence vraie classe
            ref_t = _reference_idx(t, true, preds, exclude=idx)
            if ref_t is not None:
                axes[r, 1].imshow(_load_img(test_ds, ref_t))
            axes[r, 1].set_title(f"vraie classe\n{class_names[t]}", fontsize=8, color="darkgreen")

            # référence classe prédite
            ref_p = _reference_idx(p, true, preds)
            if ref_p is not None:
                axes[r, 2].imshow(_load_img(test_ds, ref_p))
            axes[r, 2].set_title(f"classe prédite\n{class_names[p]}", fontsize=8, color="navy")

            for c in range(3):
                axes[r, c].axis("off")

        plt.tight_layout()
        plt.show()
        return

    # --- Mode grille (sans target_class) ---
    rows = (len(chosen) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3))
    axes = np.atleast_1d(axes).ravel()

    for ax, idx in zip(axes, chosen):
        ax.imshow(_load_img(test_ds, idx))
        ax.set_title(f"vrai: {class_names[true[idx]]}\nprédit: {class_names[preds[idx]]}",
                     fontsize=8, color="darkred")
        ax.axis("off")

    for ax in axes[len(chosen):]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
class_names = test_ds.classes   # ImageFolder trie par ordre alpha = ordre des labels

cm = plot_confusion_matrix(head_seathru, test_feats, test_labels, class_names)

# erreurs au hasard
show_misclassified(head_seathru, test_feats, test_labels, test_ds, class_names, n=12)



In [ ]:
# triplets pour flatworm : chaque ligne = flatworm mal classé | vrai flatworm | vraie classe prédite
show_misclassified(head_seathru, test_feats, test_labels, test_ds, class_names,
                   n=6, target_class=class_names.index("flatworm"))

In [ ]:
# triplets pour flatworm : chaque ligne = flatworm mal classé | vrai flatworm | vraie classe prédite
show_misclassified(head_seathru, test_feats, test_labels, test_ds, class_names,
                   n=6, target_class=class_names.index("fish"))

In [ ]:
# triplets pour flatworm : chaque ligne = flatworm mal classé | vrai flatworm | vraie classe prédite
show_misclassified(head_seathru, test_feats, test_labels, test_ds, class_names,
                   n=6, target_class=class_names.index("coral"))